# 03 - Finnhub Parity Check

Compares Finnhub quote fields against Polygon normalized snapshot for overlap symbols.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path('..').resolve()))
import pandas as pd

from src.common import (
    load_env, get_api_keys, load_config, new_pull_id,
    polygon_gainers, normalize_polygon_snapshot, finnhub_quote,
    normalize_finnhub_quote, save_raw_json, save_parquet
)

load_env('../.env')
cfg = load_config('../config/exploration.yaml')
keys = get_api_keys()
assert keys['polygon'], 'POLYGON_API_KEY missing in data-exploration/.env'
assert keys['finnhub'], 'FINNHUB_API_KEY missing in data-exploration/.env'


In [ ]:
pull_id = new_pull_id('parity')

poly_payload = polygon_gainers(keys['polygon'], keys['polygon_base'])
save_raw_json(poly_payload, cfg.outputs_raw_dir / f'{pull_id}_polygon_gainers.json')

df_poly = normalize_polygon_snapshot(poly_payload, '/v2/snapshot/locale/us/markets/stocks/gainers', pull_id)
symbols = df_poly['symbol'].dropna().unique().tolist()[:25]

fh_frames = []
for symbol in symbols:
    q = finnhub_quote(keys['finnhub'], keys['finnhub_base'], symbol)
    save_raw_json(q, cfg.outputs_raw_dir / f'{pull_id}_{symbol}_finnhub_quote.json')
    fh_frames.append(normalize_finnhub_quote(symbol, q, '/quote', pull_id))

df_fh = pd.concat(fh_frames, ignore_index=True) if fh_frames else pd.DataFrame()

poly_cols = ['symbol','last_price','change_pct','volume','prev_close']
fh_cols = ['symbol','last_price','change_pct','volume','prev_close']

df_cmp = df_poly[poly_cols].merge(df_fh[fh_cols], on='symbol', how='inner', suffixes=('_polygon','_finnhub'))
if not df_cmp.empty:
    df_cmp['last_price_diff_pct'] = ((df_cmp['last_price_finnhub'] - df_cmp['last_price_polygon']) / df_cmp['last_price_polygon']) * 100
    df_cmp['change_pct_diff'] = df_cmp['change_pct_finnhub'] - df_cmp['change_pct_polygon']

save_parquet(df_cmp, cfg.outputs_processed_dir / f'{pull_id}_provider_parity.parquet')
df_cmp.head()


In [ ]:
if not df_cmp.empty:
    stats = df_cmp[['last_price_diff_pct','change_pct_diff']].describe(percentiles=[0.5,0.9,0.95]).T
    display(stats)

    overlap = len(df_cmp)
    requested = len(symbols)
    print({'overlap_symbols': overlap, 'requested_symbols': requested, 'overlap_rate': overlap / requested if requested else 0})
